In [4]:
!pip install transformers datasets accelerate -q

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

In [6]:
print("Step 1: Downloading the Tokenizer and Base Model...")

model_name = "gpt2"

#tokenizer for gpt2
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 has a problem: it doesn't have a default "padding" token
# We fix this by telling it to use its "End of Text" token as the padding token.
tokenizer.pad_token = tokenizer.eos_token

#loading the model
model = AutoModelForCausalLM.from_pretrained(model_name)

print("GPT-2 is loaded")

Step 1: Downloading the Tokenizer and Base Model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 is loaded


In [7]:
print("Step 2: Downloading the AG News dataset...")

# THE DATASET
# We are downloading professional news articles from the Hugging Face hub.
# for training we are taking only 10% if the data i.e 12,000 articles
dataset = load_dataset("fancyzhx/ag_news", split="train[:10%]")

print(f"Dataset loaded! We have {len(dataset)} articles.")
print("\nExample snippet of a raw article:")
print("->", dataset[0]['text'])

Step 2: Downloading the AG News dataset...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset loaded! We have 12000 articles.

Example snippet of a raw article:
-> Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [ ]:
print("\nStep 3: Tokenizing (Translating) the text...")

# TOKENIZATION IN BATCHES
def tokenize_function(examples):
    return tokenizer(examples["text"])

# We apply (map) this function to every single article in our dataset.
# 'batched=True' means it processes multiple articles at once so it finishes in seconds instead of hours.
# We also delete (remove_columns) the original English text. Why? Because the AI only needs the numbers.
# Deleting the raw text frees up a massive amount of RAM on free Colab computer.
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

print("Data successfully translated into numbers (tokens)!")

Parameter 'function'=<function tokenize_function at 0x7ccfdd8059e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.



Step 3: Tokenizing (Translating) the text...


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Data successfully translated into numbers (tokens)!


In [10]:
print("Step 4: Formatting data into equal blocks...")

# UNIFORM DATA SIZES
# This function fixes varying lengths by mashing ALL the articles together into one giant, endless sequence,
# and then chopping that giant sequence into perfect, equal-sized blocks of 128 tokens.

def group_texts(examples):
    block_size = 128

    # Mash all the tokenized articles together into one massive list
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # Drop the tiny remainder of words at the very end so everything divides perfectly by 128
    total_length = (total_length // block_size) * block_size

    # Chop the massive list into chunks of exactly 128 tokens
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    # LABELS
    # To learn, the AI needs a "test" (input) and an "answer key" (labels).
    # Because GPT-2's only job is to predict the next word, the "answer key" is actually just
    # the exact same text! Under the hood, the AI will look at word 1 to predict word 2.
    result["labels"] = result["input_ids"].copy()

    return result

# Apply the chunking function to the data
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

print("Data formatting complete! Everything is now in blocks of 128.")

Step 4: Formatting data into equal blocks...


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

Data formatting complete! Everything is now in blocks of 128.


In [11]:
# CELL 4: SETTING UP AND RUNNING THE TRAINER
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

print("Step 5: Setting up the Training Loop...")

# The Data Collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# custom model name
custom_model_name = "./gpt2-news-anchr"

# The Rules of Learning
training_args = TrainingArguments(
    output_dir=custom_model_name,    # Saving to custom folder
    num_train_epochs=1,              # Read the dataset 1 time
    per_device_train_batch_size=8,   # Process 8 blocks at once
    learning_rate=5e-5,              # The standard step size
    save_steps=500,                  # Backup every 500 steps
    save_total_limit=1,              # Keep only 1 backup to save space
)

# The Trainer Engine
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=lm_datasets,
)

print("\n--- STARTING THE TRAINING PROCESS ---")
trainer.train()

print("\n--- TRAINING COMPLETE! ---")
# Save the final model to the temporary Colab hard drive
trainer.save_model(custom_model_name)
print(f"Model saved to temporary folder: {custom_model_name}")

Step 5: Setting up the Training Loop...

--- STARTING THE TRAINING PROCESS ---


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,3.677788


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- TRAINING COMPLETE! ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to temporary folder: ./gpt2-news-anchr


In [14]:
# THE FINAL COMPARISON TEST
from transformers import pipeline

print("Step 6: Running the Final Inference Test...\n")

# A strong, journalistic prompt to trigger the News Anchor personality
prompt = "The tech giant Apple announced a major structural change to the company today. In a press release, the CEO stated"

print("--- DIRECT COMPARISON TEST ---")
print(f"Prompt: '{prompt}'\n")

# Load a FRESH copy of the base, untrained model from the internet
print("Loading the ORIGINAL, UNTRAINED GPT-2 model...")
base_generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0  # device=0 tells it to use the GPU to generate text quickly
)

print("\n[BEFORE] Base Model Output:")
base_results = base_generator(
    prompt,
    max_new_tokens=40,      # Generate up to 40 new words
    num_return_sequences=1,
    temperature=0.7,        # A moderate creativity setting
    do_sample=True,
    truncation=True,
    pad_token_id=50256      # Hides a harmless warning message
)
print(base_results[0]['generated_text'])
print("-" * 60)


# Load the Fine-Tuned model you just created
print("\nLoading FINE-TUNED model...")
fine_tuned_generator = pipeline(
    "text-generation",
    model="./gpt2-news-anchr",
    device=0
)

print("\n[AFTER] Fine-Tuned Model Output:")
fine_tuned_results = fine_tuned_generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    truncation=True,
    pad_token_id=50256
)
print(fine_tuned_results[0]['generated_text'])
print("-" * 60)

Step 6: Running the Final Inference Test...

--- DIRECT COMPARISON TEST ---
Prompt: 'The tech giant Apple announced a major structural change to the company today. In a press release, the CEO stated'

Loading the ORIGINAL, UNTRAINED GPT-2 model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'pad_token_id', 'num_return_sequences', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[BEFORE] Base Model Output:


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The tech giant Apple announced a major structural change to the company today. In a press release, the CEO stated that Apple will be able to offer Apple's Watch for $249 in the first quarter of 2018.

The company is also announcing a new partnership with Google to offer its own smartwatch. The
------------------------------------------------------------

Loading FINE-TUNED model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[AFTER] Fine-Tuned Model Output:
The tech giant Apple announced a major structural change to the company today. In a press release, the CEO stated "We're moving quickly to ensure that our software and services are delivered in a stable and robust way."    Apple has released its latest operating system version, the iMac.   
------------------------------------------------------------
